<a href="https://colab.research.google.com/github/azizbekreimbaev/JaydatiGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformers bitsandbytes datasets peft accelerate trl "torchao>=0.16.0"

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from trl import SFTTrainer

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)


model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)


# Quantization - xotirani saqlash uchun

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_quant_type='nf4'
# )


import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
bnb_config

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     dtype=torch.bfloat16

# )


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Before Fine-Tuning
prompt = "Explain what a tokenizer is."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

print(model)
print("======================")
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is.
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
  

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## Datasets library



*   Instruction tuning



In [ ]:
from datasets import load_dataset

dataset = load_dataset('yahma/alpaca-cleaned', split='train')

In [ ]:
dataset[0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [ ]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )


In [ ]:
generate_prompt(dataset[0])

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n\n'

In [ ]:

def formatting_func(example):
    return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [ ]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n\n'

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42)

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [ ]:
# Full Fine-tuning  =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,  # rank qancha kichik bolsa shuncha kichik parametrni tuning qiladi bu memoryga gpuga yaxshi
    lora_alpha=16,  #scaling qilib beradi
    lora_dropout=0.05, #overfit bolishini oldini olish uchun
    task_type="CAUSAL_LM", # yengilloq task yoki qiyin task berishimiz mumkin
    target_modules=["q_proj", "v_proj"], #bular model attention qismida layeridan olinmoqda va bu ikkita layer eng muhim qilsmlari hisoblanadi
)

In [ ]:
model = get_peft_model(model, config)

In [ ]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# from transformers import TrainingArguments

# training_args = TrainingArguments(
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,  # VRAM
#     num_train_epochs=2,  # overfit
#     logging_steps=20,
#     output_dir="./jayadari_gpt",
#     save_strategy="epoch",
#     fp16=True,
#     report_to="none"
# )

from trl import SFTConfig, SFTTrainer


training_args = SFTConfig(
    output_dir="./jaydari_gpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=20,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    loss_type="nll",
    max_length=512,
    report_to="none"
)

In [ ]:
print(dataset.column_names)

['output', 'input', 'instruction', 'text']


In [ ]:
import importlib

# Disable unsupported CUDA mixed-dtype operation on T4
chunked_logprob = importlib.import_module(
    "trl.kernels.chunked_logprob"
)

chunked_logprob._MM_OUT_DTYPE = False

print("T4 compatibility workaround enabled")

T4 compatibility workaround enabled


In [ ]:
# T4: keep trainable LoRA parameters in float32 for FP16 GradScaler.
model.zero_grad(set_to_none=True)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)
# Check after trainer initialization, which can change adapter dtypes.
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
print("Trainable dtypes:", {p.dtype for p in model.parameters() if p.requires_grad})
trainer.train()
model.save_pretrained("jaydari_gpt")
tokenizer.save_pretrained("jaydari_gpt")

Trainable dtypes: {torch.float32}


Step,Training Loss
20,1.553781
40,1.642943
60,1.400410
80,1.509887
100,1.462233
120,1.454075
140,1.421895
160,1.277496
180,1.344798
200,1.246367


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)

model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(
    prompt,
    return_tensors="pt"
).to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL OUTPUT =====
Explain what machine learning is in simple words.


FINE-TUNED MODEL INFERENCE

In [ ]:
model_path = "jaydari_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
# model.eval()
# prompt = "### Instruction:\nExplain what machine learning is in simple words.\n\n### Response:\n"
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# with torch.inference_mode():
#     output = model.generate(**inputs, max_new_tokens=120, do_sample=False, pad_token_id=tokenizer.eos_token_id, use_cache=True)
# answer = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
# print("===== FINE-TUNED MODEL OUTPUT =====")
# print(answer)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FINE-TUNED MODEL OUTPUT =====
Machine learning is a field of computer science that involves the use of algorithms and data to learn from and make predictions based on data. It is a powerful tool for analyzing large amounts of data and identifying patterns and trends that can help businesses make informed decisions.

Machine learning algorithms work by analyzing large datasets and identifying patterns and relationships that can be used to make predictions or make decisions. They can be used to identify trends, predict future events, and make recommendations based on the data.

One of the key advantages of machine learning is its ability to


In [ ]:
model_ft = model
tokenizer_ft = tokenizer

model_ft.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [ ]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer_ft(
    prompt,
    return_tensors="pt"
).to(model_ft.device)

with torch.no_grad():
    output_ft = model_ft.generate(
        **inputs_ft,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FINE-TUNED MODEL OUTPUT =====
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning is a field of computer science that involves the development and application of algorithms and models that can optimize performance and learn from data. This is achieved through the use of statistical techniques, such as regression analysis, decision trees, neural networks, and clustering, and through the use of algorithms that can make predictions, classify data, and find patterns in large datasets. It is a powerful tool for understanding and modeling complex systems and has been used in a wide variety of applications, including healthcare, finance, marketing, and social media.




In [ ]:
!zip -r jaydari_gpt.zip jaydari_gpt

  adding: jaydari_gpt/ (stored 0%)
  adding: jaydari_gpt/checkpoint-3500/ (stored 0%)
  adding: jaydari_gpt/checkpoint-3500/optimizer.pt (deflated 8%)
  adding: jaydari_gpt/checkpoint-3500/trainer_state.json (deflated 79%)
  adding: jaydari_gpt/checkpoint-3500/scaler.pt (deflated 64%)
  adding: jaydari_gpt/checkpoint-3500/training_args.bin (deflated 53%)
  adding: jaydari_gpt/checkpoint-3500/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/checkpoint-3500/adapter_config.json (deflated 60%)
  adding: jaydari_gpt/checkpoint-3500/README.md (deflated 65%)
  adding: jaydari_gpt/checkpoint-3500/rng_state.pth (deflated 26%)
  adding: jaydari_gpt/checkpoint-3500/scheduler.pt (deflated 61%)
  adding: jaydari_gpt/checkpoint-3500/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/checkpoint-3500/adapter_model.safetensors (deflated 8%)
  adding: jaydari_gpt/checkpoint-3500/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt

MODEL TESTING

In [117]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/jaydari_gpt"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [118]:
!pip install -q langdetect

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [119]:
from langdetect import detect_langs

def ask_jaydari(prompt: str, max_new_tokens=256):
    try:
        langs = detect_langs(prompt)
        lang_code = langs[0].lang
        confidence = langs[0].prob
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")
    except:
        lang_code = "unknown"
        confidence = 0.0

    # Block only if we're confident it's NOT English
    if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
        return "Sorry, you should ask a question only in English."

    # Proceed with generation
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(full_prompt):].strip()

In [121]:
ask_jaydari('Explain about Python')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌍 Detected lang: en (1.00)


'Python is a high-level, interpreted, object-oriented programming language designed for programming on the computer. It is known for its simplicity, flexibility, and speed, making it an excellent choice for a wide range of applications. Python is an interpreted language, which means that it runs directly on the computer without requiring the use of a compiler or interpreter. This means that Python programs are run directly on the computer, rather than being translated into machine code and executed by a computer. This makes Python an ideal choice for programming on the web, as it can be easily integrated into web applications and can be run on both desktop and mobile devices. Python also supports a wide range of libraries and packages, which make it easy to integrate with other applications and tools. Python is known for its flexibility, as it is easy to learn and can be used to create a wide range of applications and projects. Python is also known for its community, which is constantl

In [123]:
ask_jaydari('Can you tell about Javascript')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌍 Detected lang: ca (0.86)


"JavaScript is a popular programming language used for web development. It is a dynamic, interpreted language that enables web developers to create dynamic and interactive web applications. The language has a simple syntax, and developers can create complex applications with ease. It supports a wide range of features, including loops, conditional statements, functions, and objects, making it an ideal language for web development.\n\nJavaScript is a server-side language, meaning that it runs on the server-side of a web application. This means that it runs on the server and communicates with the browser through HTTP requests. The language is fast, reliable, and can handle large amounts of data, making it an ideal choice for web applications that require real-time updates.\n\nJavaScript has a vast community of developers, and there are many resources available online to help learn the language. One of the most popular resources is the official documentation for the language, which include

In [125]:
ask_jaydari('Quyosh va Mars farqi nima')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌍 Detected lang: sq (0.57)


'The following is the translation of the phrase "Quyosh va Mars farqi nima" into English:\n\n"The day of the Quyosh and Mars is approaching."'